## Demo 2: StackExchange

The data of a StackExchange site, published as one XML file per table on archive.org.

This demo is about getting files into databases, and about moving data from one database into another one. We start by looking at the data, without touching a database at all.

### What data do we have?

In [20]:
import os

data_path = r"..\data\stackexchange"

for file in sorted(os.listdir(data_path)):
    size = os.path.getsize(os.path.join(data_path, file))
    print(f"{file:20} {size / 1024 / 1024:6.1f} MB")

Badges.xml              2.0 MB
Comments.xml            1.3 MB
PostHistory.xml         7.6 MB
PostLinks.xml           0.0 MB
Posts.xml               3.5 MB
README.md               0.0 MB
Tags.xml                0.0 MB
Users.xml               4.7 MB
Votes.xml               1.5 MB


### The files are XML, but they are also line oriented

That is what makes them pleasant to work with: the whole file is valid XML, and every single row is valid XML on its own. So we never have to load the whole document into memory.

In [21]:
users_path = r"..\data\stackexchange\Users.xml"

with open(users_path, encoding="utf-8-sig") as file:
    users_lines = file.readlines()

print(len(users_lines), "lines")
print(users_lines[0], end="")
print(users_lines[1], end="")
print(users_lines[2][:110], "...")
print(users_lines[-1])

12223 lines
<?xml version="1.0" encoding="utf-8"?>
<users>
  <row Id="-1" Reputation="1" CreationDate="2011-01-03T17:13:13.000" DisplayName="Community" LastAccessDate="2 ...
</users>


#### Why `utf-8-sig` and not `utf-8`?

The file starts with a byte order mark. `Get-Content` in PowerShell removes it without telling us, `open` in Python does not. With plain `utf-8` the first line starts with an invisible `\ufeff`, and a test like `line.startswith("<?xml")` is suddenly false.

In [22]:
for encoding in ["utf-8", "utf-8-sig"]:
    with open(users_path, encoding=encoding) as file:
        first_line = file.readline()

    print(f"{encoding:10} {first_line[:24]!r:36} starts with '<?xml': {first_line.startswith('<?xml')}")

utf-8      '\ufeff<?xml version="1.0" enc'      starts with '<?xml': False
utf-8-sig  '<?xml version="1.0" enco'           starts with '<?xml': True


### One line is one row

In PowerShell we cast the line to `[xml]` and get an `XmlElement` with one property per attribute. In Python we parse the line and take its attributes, and what we get is a plain dictionary.

In [23]:
import xml.etree.ElementTree as ET

line = users_lines[2]

row = ET.fromstring(line).attrib

row

{'Id': '-1',
 'Reputation': '1',
 'CreationDate': '2011-01-03T17:13:13.000',
 'DisplayName': 'Community',
 'LastAccessDate': '2011-01-03T17:13:19.040',
 'WebsiteUrl': 'http://meta.stackexchange.com/',
 'Location': 'on the server farm',
 'AboutMe': '<p>Hi, I\'m not really a person.</p>\n<p>I\'m a background process that helps keep this site clean!</p>\n<p>I do things like</p>\n<ul>\n<li>Randomly poke old unanswered questions every hour so they get some attention</li>\n<li>Own community questions and answers so nobody gets unnecessary reputation from them</li>\n<li>Own downvotes on spam/evil posts that get permanently deleted</li>\n<li>Own suggested edits from anonymous users</li>\n<li><a href="http://meta.stackexchange.com/a/92006">Remove abandoned questions</a></li>\n</ul>\n',
 'Views': '185',
 'UpVotes': '0',
 'DownVotes': '0',
 'AccountId': '-1'}

In [24]:
print(type(row))
print(type(row["Id"]), repr(row["Id"]))

<class 'dict'>
<class 'str'> '-1'


Every value is a string, in both languages. Nobody has converted anything yet - the types will come from the target table.

### Not every row has every attribute

And this is where the two languages really differ. An attribute that is not in the line is simply not in the dictionary.

In [25]:
from collections import Counter

attribute_counts = Counter()
row_count = 0

for line in users_lines:
    if line.lstrip().startswith("<row"):
        row_count += 1
        attribute_counts.update(ET.fromstring(line).attrib.keys())

for attribute, count in attribute_counts.most_common():
    print(f"{attribute:16} {count:6} of {row_count}")

Id                12220 of 12220
Reputation        12220 of 12220
CreationDate      12220 of 12220
DisplayName       12220 of 12220
LastAccessDate    12220 of 12220
Views             12220 of 12220
UpVotes           12220 of 12220
DownVotes         12220 of 12220
AccountId         12220 of 12220
Location           6813 of 12220
AboutMe            5824 of 12220
WebsiteUrl         3927 of 12220


In [26]:
# Find the first user without a Location

for line in users_lines:
    if line.lstrip().startswith("<row"):
        sparse_row = ET.fromstring(line).attrib
        if "Location" not in sparse_row:
            break

print(sparse_row["DisplayName"], "has no Location")

# PowerShell gives us $null for a missing property, and so does .get()
print(sparse_row.get("Location"))

REW has no Location
None


In [27]:
# But asking for it directly is an error, not a None

sparse_row["Location"]

KeyError: 'Location'

So a Python port of the import cannot simply read `row[column]` for every column of the target table. It either asks with `.get()`, or it has to know which attributes are there.

### From lines to a data frame

pandas has its own answer to the missing attributes: it collects every key it sees and fills the gaps with `NaN`.

In [28]:
import pandas as pd

users = pd.DataFrame(
    ET.fromstring(line).attrib
    for line in users_lines
    if line.lstrip().startswith("<row")
)

users

,Id,Reputation,CreationDate,DisplayName,LastAccessDate,WebsiteUrl,Location,AboutMe,Views,UpVotes,DownVotes,AccountId
0,-1,1,2011-01-03T17:13:13.000,Community,2011-01-03T17:13:19.040,http://meta.stackexchange.com/,on the server farm,"<p>Hi, I'm not really a person.</p>\n<p>I'm a ...",185,0,0,-1
1,2,101,2011-01-03T20:14:55.000,Geoff Dalgas,2016-04-04T15:28:18.677,http://stackoverflow.com,"Corvallis, OR",<p>Dev #2 who helped create Stack Overflow cur...,20,0,0,2
2,3,101,2011-01-03T20:15:50.000,balpha,2015-08-19T13:32:48.957,https://social.balpha.de/@balpha,"Berlin, Germany",<p>My name is Benjamin Dumke-von der Ehe. I wo...,29,0,0,40051
3,4,212,2011-01-03T20:18:17.000,Nick Craver,2019-07-09T22:33:34.993,https://nickcraver.com/blog/,"Winston-Salem, NC",<p>I am a Principal Software Engineer at Micro...,5,2,1,7598
4,5,101,2011-01-03T20:26:51.000,Emmett,2012-04-16T21:48:25.827,http://minesweeperonline.com,"San Francisco, CA","<p>co-founder of <a href=""https://airtable.com...",1,0,0,1998
...,...,...,...,...,...,...,...,...,...,...,...,...
12215,288569,101,2024-03-23T13:50:08.000,liquidcms,2024-03-23T14:34:04.547,http://www.liquidcms.ca,Canada,<p>Drupal developer since 2006. Ionic develope...,0,0,0,4218730
12216,288631,1,2024-03-25T13:28:08.000,Myran thiru,2024-03-28T13:22:15.420,NaN,NaN,NaN,0,0,0,19055225
12217,288670,1,2024-03-26T10:42:42.000,Ekemini,2024-03-26T12:03:56.540,NaN,NaN,NaN,0,0,0,30998810
12218,288770,1,2024-03-28T14:02:06.000,ldhwaddell,2024-03-28T16:42:42.903,NaN,NaN,NaN,0,0,0,26685757


In [29]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 12220 entries, 0 to 12219
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Id              12220 non-null  str  
 1   Reputation      12220 non-null  str  
 2   CreationDate    12220 non-null  str  
 3   DisplayName     12220 non-null  str  
 4   LastAccessDate  12220 non-null  str  
 5   WebsiteUrl      3927 non-null   str  
 6   Location        6813 non-null   str  
 7   AboutMe         5824 non-null   str  
 8   Views           12220 non-null  str  
 9   UpVotes         12220 non-null  str  
 10  DownVotes       12220 non-null  str  
 11  AccountId       12220 non-null  str  
dtypes: str(12)
memory usage: 1.1 MB


Twelve columns, all of them strings, and three of them with missing values. Converting those strings into the types of a database table is the next step.

### Setting up the connection to SQL Server

Same three lines as in the first demo, only the database and the login are different.

In [30]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_sql_instance import connect_sql_instance
from import_sql_table import import_sql_table
from invoke_sql_query import invoke_sql_query

connection = connect_sql_instance(
    instance="127.0.0.1",
    database="StackExchange",
    username="StackExchange",
    password="Passw0rd!"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using SQL authentication
[VERBOSE] Disabling connection pooling
[VERBOSE] Opening connection
[VERBOSE] Returning connection object


### What does the target table look like?

The file gives us strings. The table decides what they have to become. `cursor.description` tells us the columns and, for each one, the Python type that pyodbc expects. It is what `GetSchemaTable()` is for the PowerShell version.

In [31]:
cursor = connection.cursor()
cursor.execute("SELECT TOP 0 * FROM dbo.Users")
description = cursor.description
cursor.close()

for name, type_code, _, _, _, _, null_ok in description:
    print(f"{name:16} {type_code.__name__:10} null_ok={null_ok}")

Id               int        null_ok=False
AboutMe          str        null_ok=True
Age              int        null_ok=True
CreationDate     datetime   null_ok=False
DisplayName      str        null_ok=True
DownVotes        int        null_ok=False
EmailHash        str        null_ok=True
LastAccessDate   datetime   null_ok=False
Location         str        null_ok=True
Reputation       int        null_ok=False
UpVotes          int        null_ok=False
Views            int        null_ok=False
WebsiteUrl       str        null_ok=True
AccountId        int        null_ok=True


#### The file and the table do not agree

The table has fourteen columns, a row in the file has twelve attributes, and not even the same twelve on every row.

In [32]:
table_columns = [column[0] for column in description]

print("in the table but never in the file:", [c for c in table_columns if c not in attribute_counts])
print("in the file but not in the table:  ", [a for a in attribute_counts if a not in table_columns])

in the table but never in the file: ['Age', 'EmailHash']
in the file but not in the table:   []


### Importing the file

`import_sql_table` reads the file line by line, so the size of the file does not matter. For every line it builds one value per column of the *target* table: it asks the row with `.get()`, so a missing attribute becomes `NULL`, converts the string with the type from `cursor.description`, and sends the rows to the database in batches.

In [33]:
import_sql_table(
    connection=connection,
    path=r"..\data\stackexchange\Users.xml",
    table="dbo.Users",
    batch_size=5000,
    truncate_table=True
)

[VERBOSE] Importing data from ..\data\stackexchange\Users.xml into [dbo].[Users]
[VERBOSE] Creating cursor
[VERBOSE] Truncating table
[VERBOSE] Inserting rows
[VERBOSE] 5000 rows inserted (48.3%) - 8108 rows/sec
[VERBOSE] 10000 rows inserted (86.0%) - 7805 rows/sec
[VERBOSE] Imported 12220 rows in 1.6 seconds


In [34]:
invoke_sql_query(
    connection=connection,
    query='SELECT TOP 5 Id, DisplayName, Location, CreationDate, Reputation FROM dbo.Users'
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,Id,DisplayName,Location,CreationDate,Reputation
0,-1,Community,on the server farm,2011-01-03 17:13:13,1
1,2,Geoff Dalgas,"Corvallis, OR",2011-01-03 20:14:55,101
2,3,balpha,"Berlin, Germany",2011-01-03 20:15:50,101
3,4,Nick Craver,"Winston-Salem, NC",2011-01-03 20:18:17,212
4,5,Emmett,"San Francisco, CA",2011-01-03 20:26:51,101


The dates are dates and the numbers are numbers, and the two columns that the file never mentions are `NULL` for every row - just like the `Location` of the users that did not fill it in.

In [35]:
invoke_sql_query(connection=connection, query="""
SELECT COUNT(*) AS ImportedRows,
       SUM(CASE WHEN Location IS NULL THEN 1 ELSE 0 END) AS NullLocation,
       SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS NullAge
FROM dbo.Users""")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 1 rows


,ImportedRows,NullLocation,NullAge
0,12220,5407,12220


### When the file names a column differently

Badges are created on `Date`, but every table in this database calls that column `CreationDate`.

In [36]:
badges_path = r"..\data\stackexchange\Badges.xml"

with open(badges_path, encoding="utf-8-sig") as file:
    badges_lines = file.readlines()

ET.fromstring(badges_lines[2]).attrib

{'Id': '13',
 'UserId': '6',
 'Name': 'Teacher',
 'Date': '2011-01-03T21:19:04.180',
 'Class': '3',
 'TagBased': 'False'}

In [37]:
import_sql_table(
    connection=connection,
    path=badges_path,
    table="dbo.Badges",
    batch_size=5000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

[VERBOSE] Importing data from ..\data\stackexchange\Badges.xml into [dbo].[Badges]
[VERBOSE] Creating cursor
[VERBOSE] Truncating table
[VERBOSE] Inserting rows
[VERBOSE] 5000 rows inserted (26.9%) - 15579 rows/sec
[VERBOSE] 10000 rows inserted (54.3%) - 14517 rows/sec
[VERBOSE] 15000 rows inserted (80.5%) - 13409 rows/sec
[VERBOSE] Imported 18707 rows in 1.5 seconds


In [38]:
invoke_sql_query(connection=connection, query='SELECT TOP 5 * FROM dbo.Badges')

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,Id,Name,UserId,CreationDate
0,1,Autobiographer,2,2011-01-03 20:24:04.670
1,2,Autobiographer,21,2011-01-03 20:49:04.763
2,3,Autobiographer,25,2011-01-03 20:54:04.180
3,4,Autobiographer,38,2011-01-03 20:59:04.283
4,5,Student,24,2011-01-03 21:04:04.183


### The one thing that is really different

The PowerShell version fills a `DataTable` whose columns are typed from `GetSchemaTable()`, and lets ADO.NET convert the strings on the way in. pyodbc has nothing like that: in fast bulk mode it binds a value by its Python type, so a string never reaches an `INT` column.

So `import_sql_table` carries a small table of converters - `int`, `str`, `datetime.fromisoformat` - and picks one per column from `cursor.description`. That table is the part of this function with no counterpart in the sibling repository.

### The same file, into PostgreSQL

Now the second database. The functions are deliberately named and shaped like the SQL Server ones, so a call site looks almost the same. Two things underneath are genuinely different.

In [39]:
from connect_pg_instance import connect_pg_instance
from import_pg_table import import_pg_table
from invoke_pg_query import invoke_pg_query

pg_connection = connect_pg_instance(
    instance="127.0.0.1",
    database="stackexchange",
    username="stackexchange",
    password="Passw0rd!"
)

[VERBOSE] Creating connection to instance [127.0.0.1]
[VERBOSE] Using password authentication
[VERBOSE] Opening connection
[VERBOSE] Returning connection object


#### First difference: PostgreSQL folds identifiers to lower case

The tables are created as `CREATE TABLE Users (Id INT ...)` without quotes, so the catalog holds `users` and `id`. The attributes in the file are `Id`, `AboutMe`, `CreationDate`.

In [40]:
cursor = pg_connection.cursor()
cursor.execute("SELECT * FROM Users WHERE 1=0")
pg_columns = [column.name for column in cursor.description]
cursor.close()

print("columns in PostgreSQL:", pg_columns)
print()
print("exact matches with the XML attributes:", [c for c in pg_columns if c in attribute_counts])
print("case insensitive matches:             ",
      [c for c in pg_columns if c.lower() in {a.lower() for a in attribute_counts}])

columns in PostgreSQL: ['id', 'aboutme', 'age', 'creationdate', 'displayname', 'downvotes', 'emailhash', 'lastaccessdate', 'location', 'reputation', 'upvotes', 'views', 'websiteurl', 'accountid']

exact matches with the XML attributes: []
case insensitive matches:              ['id', 'aboutme', 'creationdate', 'displayname', 'downvotes', 'lastaccessdate', 'location', 'reputation', 'upvotes', 'views', 'websiteurl', 'accountid']


Not a single column matches by name.

In PowerShell this never comes up: `$row.aboutme` finds the `AboutMe` attribute, because property access is case insensitive. A Python dictionary is not. So both import functions lower case the keys of a row and the names of the columns before matching them.

Without that, the import would write 12220 rows of `NULL` and report success - no error anywhere.

In [41]:
import_pg_table(
    connection=pg_connection,
    path=r"..\data\stackexchange\Users.xml",
    table="Users",
    batch_size=5000,
    truncate_table=True
)

[VERBOSE] Importing data from ..\data\stackexchange\Users.xml into "users"
[VERBOSE] Creating cursor
[VERBOSE] Truncating table
[VERBOSE] Inserting rows
[VERBOSE] 5000 rows inserted (48.3%) - 6559 rows/sec
[VERBOSE] 10000 rows inserted (86.0%) - 6766 rows/sec
[VERBOSE] Imported 12220 rows in 1.7 seconds


In [42]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, displayname, location, creationdate, reputation FROM Users LIMIT 5"
)

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,id,displayname,location,creationdate,reputation
0,-1,Community,on the server farm,2011-01-03 17:13:13,1
1,2,Geoff Dalgas,"Corvallis, OR",2011-01-03 20:14:55,101
2,3,balpha,"Berlin, Germany",2011-01-03 20:15:50,101
3,4,Nick Craver,"Winston-Salem, NC",2011-01-03 20:18:17,212
4,5,Emmett,"San Francisco, CA",2011-01-03 20:26:51,101


#### Second difference: COPY instead of INSERT

`import_sql_table` builds an `INSERT` and sends batches through `executemany`, and it has to convert every string into the type of its column first, because pyodbc binds a value by its Python type.

PostgreSQL has `COPY`, which takes the text and parses it into the column type itself. So `import_pg_table` carries no converters at all. Measured on this file, 12220 rows:

| | |
| --- | --- |
| `executemany`, converted values | 1.27 s |
| `executemany`, raw strings | 0.95 s |
| `COPY`, converted values | 0.30 s |
| `COPY`, raw strings | 0.14 s |

The fastest way is also the one with the least code - the opposite of SQL Server, where passing raw strings fails outright.

In [43]:
import_pg_table(
    connection=pg_connection,
    path=badges_path,
    table="Badges",
    batch_size=10000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

[VERBOSE] Importing data from ..\data\stackexchange\Badges.xml into "badges"
[VERBOSE] Creating cursor
[VERBOSE] Truncating table
[VERBOSE] Inserting rows
[VERBOSE] 10000 rows inserted (54.3%) - 21007 rows/sec
[VERBOSE] Imported 18707 rows in 1.2 seconds


In [44]:
invoke_pg_query(connection=pg_connection, query="SELECT * FROM Badges LIMIT 5")

[VERBOSE] Creating cursor
[VERBOSE] Executing query
[VERBOSE] Retrieved 5 rows


,id,name,userid,creationdate
0,13,Teacher,6,2011-01-03 21:19:04.180
1,14,Teacher,41,2011-01-03 21:19:04.353
2,32,Teacher,43,2011-01-04 01:09:04.050
3,55,Teacher,126,2011-01-04 06:24:04.097
4,60,Teacher,136,2011-01-04 07:09:04.353


The same call with the same parameters against a second database system - and underneath, one function converts every value by hand while the other hands the text straight to the database.

### Moving data between databases

Every import so far read a file. Now the source is another table - and it does not have to be in the same database, or even in the same database system.

#### What is a data reader here?

`Get-SqlDataReader` in PowerShell returns a `DbDataReader`: an open result set that hands out one row at a time, so the rows never all sit in memory at once.

In Python a cursor already is exactly that. `get_sql_data_reader` runs the query and returns the cursor, the writer reads it in batches with `fetchmany`, and the writer closes it when it is done - the same ownership as in the sibling, where `Write-SqlTable` disposes the reader it was handed.

In [ ]:
from get_pg_data_reader import get_pg_data_reader
from get_sql_data_reader import get_sql_data_reader
from write_pg_table import write_pg_table
from write_sql_table import write_sql_table

#### Inside one database system

The source gets its own connection, because one connection is reading while the other is writing. The row count is passed in separately: a reader does not know how many rows are still coming, so without it there is no percentage to show.

In [ ]:
source_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="StackExchange",
    username="StackExchange",
    password="Passw0rd!"
)

row_count = invoke_sql_query(
    connection=source_connection,
    query="SELECT COUNT(*) FROM dbo.Users",
    as_type="single_value"
)

data_reader = get_sql_data_reader(connection=source_connection, table="dbo.Users")

write_sql_table(
    connection=connection,
    table="dbo.Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

# The writer closed the reader, but the connection we opened is ours to close. A reader keeps
# its transaction open, and a connection left in one keeps its locks.
source_connection.close()

#### From PostgreSQL into SQL Server

The same two calls. Only the function that opens the reader is a different one, and the rows now cross from one database system into another.

In [ ]:
row_count = invoke_pg_query(
    connection=pg_connection,
    query="SELECT COUNT(*) FROM Users",
    as_type="single_value"
)

data_reader = get_pg_data_reader(connection=pg_connection, table="Users")

write_sql_table(
    connection=connection,
    table="dbo.Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

In [ ]:
invoke_sql_query(
    connection=connection,
    query="SELECT TOP 5 Id, DisplayName, Location, CreationDate FROM dbo.Import_Users"
)

Nothing had to be converted on the way. psycopg hands out Python objects, pyodbc takes Python objects, and the dates, numbers and `NULL`s arrive as what they were.

#### And back the other way

Same shape again, and this one is the fastest of the four, because the target is PostgreSQL and `write_pg_table` feeds the rows straight into `COPY`.

In [ ]:
row_count = invoke_sql_query(
    connection=connection,
    query="SELECT COUNT(*) FROM dbo.Users",
    as_type="single_value"
)

data_reader = get_sql_data_reader(connection=connection, table="dbo.Users")

write_pg_table(
    connection=pg_connection,
    table="Import_Users",
    data_reader=data_reader,
    data_reader_row_count=row_count,
    batch_size=2000,
    truncate_table=True
)

In [ ]:
invoke_pg_query(
    connection=pg_connection,
    query="SELECT id, displayname, location, creationdate FROM Import_Users LIMIT 5"
)

All four directions of the same 12220 rows, measured on this machine:

| | |
| --- | --- |
| SQL Server to SQL Server | 1.00 s |
| PostgreSQL to SQL Server | 0.88 s |
| SQL Server to PostgreSQL | 0.31 s |
| PostgreSQL to PostgreSQL | 0.16 s |

The two that write into PostgreSQL are the quick ones, for the same reason the file import was: `COPY`. And in none of them was the whole table ever in memory - the writer asked the reader for the next batch, wrote it, and asked again.